# Ejemplo de datos con censura

Datos de consumo de gas LP, ¿cuánto Gas LP consumen los hogares en México?, ¿tiene sustitutos?

## 1. Importación de bibliotecas y datos:

In [ ]:
import pandas as pd
import statsmodels.api as sm
import numpy as np
import matplotlib.pyplot as plt
from tobit import *

# Omiting WARNINGS
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Read data
GasLP_df = pd.read_stata('Gas_LP.dta')
GasLP_df

* folioviv: Identificador de la vivienda
* foliohog: Identificador del hogar
* ageb: Área geoestadísstica básica
* est_dis: Estrato de diseño muestral
* upm: Unidad primaria de muestreo
* factor: Factor de expansión
* ubica_geo: Ubicación geográfica
* entidad: Entidad
* cve_ent:               
* Entidad: Entidad
* cve_mun     
* nom_mun: nom_mun
* cve_loc: 
* lat_dec_loc: LAT_DEC
* lon_dec_loc: LON_DEC
* altitud: ALTITUD
* tam_loc: Tamaño de localidad
* rural: Rural=1 según folioviv
* tot_integ: Número de integrantes del hogar
* estufa: Tiene estufa en su casa
* calent_sol: Dispone de calentador solar de agua
* tanque_gas: Dispone de tanque de gas estacionario
* tenencia: Tipo de tenencia de la vivienda
* rentada: Vivienda rentada o prestada
* ing_cor: Ingreso corriente en miles
* ing_deciles: # de decil de ingreso
* ling_cor: Log del ingreso corriente mensual
* ling_cor2: Cuadrado del log del ingreso corriente mensual
* gasto_gaslp: Gasto en gas LP
* p_gaslp: Precio por kg del gas LP
* lgasto_gaslp: log de gasto en gas LP
* lp_gaslp: Log de precio por kg de gaslp
* gasto_elect: Gasto en electricdad
* p_elect: Precio en centavos por Kw/hr de la electricidad
* gasto_gasnat: Gasto en gas natural
* p_gasnat: Precio del gas natural en pesos por GJ
* lp_gasnat: Log del precio por GJ de gas natural
* lp_gasnat_kg: Log del precio por kg equivalnte del gas natural
* gasto_carb: Gasto en carbon
* gasto_lena: Gasto en legna
* gasto_perol: Gasto en petróleo en hogar
* gasto_diesel: Gasto en diesel
* gasto_comb: Gasto en combustible para calentar
* gasto_otroscomb: Gasto en otros combustibles
* acc_gasnat: Dummy, =1 si su localidad tiene acceso a gas natural, =0 si no

Fuente: ENIGH-2016 y otras fuentes del Gobierno Federal

## 2. Análisis descriptivo

¿El gasto en Gas LP es una variabe censurada?

In [ ]:
#
GasLP_df.lgasto_gaslp

In [ ]:
# Remplazamos NaN con 0
GasLP_df['lgasto_gaslp'] = GasLP_df['lgasto_gaslp'].fillna(0)

> **Cuidado con `inplace=True`.** La forma `GasLP_df['lgasto_gaslp'].fillna(0, inplace=True)` opera sobre una copia intermedia: en pandas 2 todavía funciona (con una advertencia), pero **en pandas 3 no modifica nada** y no avisa. Los `NaN` sobreviven, el `dropna` posterior elimina a todos los hogares con gasto cero, y el Tobit se estima sin una sola observación censurada. Por eso se reasigna la columna explícitamente.

In [ ]:
#
GasLP_df.lgasto_gaslp

In [ ]:
# plotting histogram in order to see
plt.hist(GasLP_df.gasto_gaslp, density = True, histtype='bar')

#plt.legend()
plt.ylabel('Gasto en Gas LP')
plt.title('Histogram')

plt.show()

In [ ]:
#
GasLP_df2 = GasLP_df[GasLP_df.gasto_gasnat == 0]

GasLP_df2

In [ ]:
#

GasLP_df2. columns

In [ ]:
# Borramos valores NaN de la muestra
GasLP_df2 = GasLP_df2.dropna(how = 'any')

GasLP_df2

In [ ]:
# Selección de datos:
Gasto_Positivo = GasLP_df2.loc[GasLP_df2.gasto_gaslp >  0, :]
Gasto_Positivo

In [ ]:
# plotting histogram in order to see
plt.hist(Gasto_Positivo.gasto_gaslp, density = True, histtype='bar')

#plt.legend()
plt.ylabel('Gasto en Gas LP')
plt.title('Histogram para Gasto Positivo')

plt.show()

In [ ]:
# Selección de datos:
Gasto_Medio = GasLP_df.loc[(GasLP_df.gasto_gaslp >  0) & (GasLP_df.gasto_gaslp <  700), :]
Gasto_Medio

In [ ]:
# plotting histogram in order to see
plt.hist(Gasto_Medio.gasto_gaslp, density = True, histtype='bar')

#plt.legend()
plt.ylabel('Gasto en Gas LP')
plt.title('Histogram para Gasto Medio')

plt.show()

In [ ]:
# Selección de datos:

GasLP_df3 = GasLP_df2.loc[GasLP_df2.gasto_gaslp <  700, :]

GasLP_df3

In [ ]:
# plotting histogram in order to see
plt.hist(GasLP_df3.gasto_gaslp, density = True, histtype='bar')

#plt.legend()
plt.ylabel('Gasto en Gas LP')
plt.title('Histogram para Gasto Sin Datos Aberrantes')

plt.show()

In [ ]:
# Scatter Plot 1:
# Getting x and y values
y_values = GasLP_df3['gasto_gaslp']
x_values = GasLP_df3['ing_cor']

# Plot scatter plot
plt.scatter(x_values, y_values, marker = 'o')
#plt.grid()

plt.title('Relación entre Gasto en gas LP e Ingreso (Toda la Muestra)')
plt.ylabel('Gasto')
plt.xlabel('Ingreso')

# Save the Figure
#plt.savefig("Fig1.png")

# Show plot
plt.show()

In [ ]:
# Scatter Plot 2:
# Getting x and y values
y_values = GasLP_df3['gasto_gaslp']
x_values = GasLP_df3['ing_cor']

# Plot scatter plot
plt.scatter(x_values, y_values, marker = 'o')
#plt.grid()

plt.title('Relación entre Gasto en gas LP e Ingreso (Gasto Positivo)')
plt.ylabel('Gasto')
plt.xlabel('Ingreso')

# Save the Figure
#plt.savefig("Fig1.png")

# Show plot
plt.show()

In [ ]:
# Scatter Plot 3:
# Getting x and y values
y_values = GasLP_df3['gasto_gaslp']
x_values = GasLP_df3['tot_integ']

# Plot scatter plot
plt.scatter(x_values, y_values, marker = 'o')
#plt.grid()

plt.title('Relación entre Gasto en gas LP e Ingreso (Gasto Positivo)')
plt.ylabel('Gasto')
plt.xlabel('Número Total de Integrantes del Hogar')

# Save the Figure
#plt.savefig("Fig1.png")

# Show plot
plt.show()

## 3. Análisis regresión

In [ ]:
#

GasLP_df3['cens'] = 0
GasLP_df3.loc[(GasLP_df3.lgasto_gaslp == 0), 'cens'] = -1
GasLP_df3

In [ ]:
#
# Variable de censura
cens = pd.Series(GasLP_df3.cens)

cens.value_counts(normalize = True)

> **Verificación.** Antes de estimar, conviene confirmar que la muestra contiene observaciones censuradas: aquí deben ser alrededor de 39 % (el valor `-1`). Si la proporción de `-1` fuera cero, el Tobit se reduciría exactamente a MCO — el optimizador terminaría en cero iteraciones — y la estimación no diría nada nuevo.

In [ ]:
# Definimos variables
y = GasLP_df3.lgasto_gaslp
x = GasLP_df3[['ling_cor', 'ling_cor2', 'tot_integ', 'lp_gaslp', 'acc_gasnat']]

In [ ]:
# Estimación del modelo Tobit

tr = TobitModel()
tr = tr.fit(x, y, cens, verbose = True)

In [ ]:
dir(tr)

In [ ]:
# Resultados
tr.intercept_, tr.coef_

In [ ]:
tr.sigma_

In [ ]:
# Adecuaciones a X:
x = sm.add_constant(x)

In [ ]:
# OLS
model = sm.OLS(y, x)
results = model.fit()
print(results.summary())

In [ ]:
# Comparemos Tobit y MCO lado a lado:
comparacion = pd.DataFrame({'Tobit': np.r_[tr.intercept_, tr.coef_],
                            'MCO': results.params.values},
                           index = results.params.index)
comparacion

In [ ]:
# OLS
results.params[1:6]

**Lectura de la comparación.** MCO sobre toda la muestra, con los ceros incluidos, **atenúa** todos los efectos respecto del Tobit: el del logaritmo del precio (`lp_gaslp`) pasa de alrededor de −7.1 a −4.3, y el del ingreso (`ling_cor`) de 6.2 a 3.3. Es la dirección del sesgo que anticipa el cap. 8 de las notas: la masa de ceros aplana la pendiente. Recuérdese además que los coeficientes del Tobit se refieren a la variable **latente**; para responder cuánto cambia el gasto observado hay que calcular el efecto marginal correspondiente a la esperanza que interese.

## 4. Regresión truncada: horas trabajadas de mujeres casadas

El gas LP ilustra la **censura**: los hogares con gasto cero están en la muestra, con su valor en el límite. Para contrastarla con el **truncamiento** conviene un ejemplo en el que se puedan aplicar ambos al mismo problema. Usamos los datos de Mroz (1987) sobre las horas trabajadas en 1975 por 753 mujeres casadas, de las cuales 325 no trabajaron (`hours = 0`):

* **Censura** — se conservan las 753 observaciones, sabiendo que 325 están en el límite. Se estima con **Tobit**.
* **Truncamiento** — sólo se observan las 428 mujeres que trabajan; las demás no están en la muestra. Se estima con la **regresión truncada**, cuya contribución a la log-verosimilitud, con truncamiento por abajo en cero, es

$$\ell_i(\boldsymbol\beta, \sigma) = \ln \phi\!\left(\frac{y_i - \mathbf{x}_i\boldsymbol\beta}{\sigma}\right) - \ln \sigma - \ln \Phi\!\left(\frac{\mathbf{x}_i\boldsymbol\beta}{\sigma}\right)$$

El último término reescala la densidad para que integre uno sobre la región observada, $y_i > 0$ (cap. 8 de las notas).

In [ ]:
# Datos de Mroz (1987), incluidos en linearmodels
from linearmodels.datasets import mroz

mroz_df = mroz.load()
x_mroz = ['nwifeinc', 'educ', 'exper', 'expersq', 'age', 'kidslt6', 'kidsge6']

# Tobit con la muestra completa: las horas iguales a cero están censuradas por la izquierda
cens_mroz = pd.Series(np.where(mroz_df['hours'] == 0, -1, 0), index = mroz_df.index)
tobit_mroz = TobitModel().fit(mroz_df[x_mroz], mroz_df['hours'].astype(float), cens_mroz)

print('Observaciones censuradas:', (cens_mroz == -1).sum(), 'de', len(cens_mroz))
print('Tobit - constante:', round(tobit_mroz.intercept_, 2), '  sigma:', round(tobit_mroz.sigma_, 2))
pd.Series(tobit_mroz.coef_, index = x_mroz).round(2)

> **Verificación.** Estos valores reproducen el ejemplo 17.2 de Wooldridge, *Introductory Econometrics*: constante 965.31, `educ` 80.65, `kidslt6` −894.02 y $\hat\sigma = 1{,}122.02$. De paso confirman que `tobit.py` está bien programado.

In [ ]:
from scipy.stats import norm
from statsmodels.base.model import GenericLikelihoodModel

class RegresionTruncada(GenericLikelihoodModel):
    """Regresión truncada por abajo en cero. El último parámetro es ln(sigma)."""
    def loglikeobs(self, params):
        beta, sigma = params[:-1], np.exp(params[-1])
        xb = self.exog @ beta
        return norm.logpdf((self.endog - xb) / sigma) - np.log(sigma) - norm.logcdf(xb / sigma)

# Muestra truncada: sólo las mujeres que trabajan
trabajan = mroz_df[mroz_df['hours'] > 0]
X_trunc = sm.add_constant(trabajan[x_mroz])
y_trunc = trabajan['hours'].astype(float)

# MCO sobre la muestra truncada, que además da los valores iniciales
mco_trunc = sm.OLS(y_trunc, X_trunc).fit()
inicio = np.r_[mco_trunc.params.values, np.log(np.sqrt(mco_trunc.scale))]

# Máxima verosimilitud: Nelder-Mead para acercarse y BFGS para afinar
# (BFGS solo se detiene antes del máximo con estos datos, por la escala de las horas)
modelo_trunc = RegresionTruncada(y_trunc, X_trunc, extra_params_names = ['log_sigma'])
res_trunc = modelo_trunc.fit(start_params = inicio, method = 'nm', maxiter = 50000, disp = 0)
res_trunc = modelo_trunc.fit(start_params = res_trunc.params, method = 'bfgs', maxiter = 20000, disp = 0)

print('N truncada:', len(y_trunc), '  sigma:', round(float(np.exp(np.asarray(res_trunc.params)[-1])), 2))
print('Gradiente máximo en el óptimo:', f"{np.abs(modelo_trunc.score(np.asarray(res_trunc.params))).max():.1e}")

pd.DataFrame({'MCO (truncada)': mco_trunc.params.values,
              'Regresión truncada': np.asarray(res_trunc.params)[:-1],
              'ee': np.asarray(res_trunc.bse)[:-1],
              'Tobit (censurada)': np.r_[tobit_mroz.intercept_, tobit_mroz.coef_]},
             index = X_trunc.columns).round(2)

**Lectura del cuadro.**

1. **MCO sobre la muestra truncada atenúa los efectos**: el de tener hijos pequeños (`kidslt6`) es de −306 horas con MCO y de −485 con la regresión truncada. Es la dirección del sesgo que anticipa el cap. 8: al quedarse sólo con quienes trabajan, se pierden precisamente las observaciones que más reaccionan.
2. **Si el modelo Tobit fuera correcto, la regresión truncada y el Tobit estimarían los mismos parámetros**, porque ambos parten de la misma variable latente; el primero sólo usa menos información. Aquí difieren mucho: el coeficiente de la educación incluso cambia de signo (80.7 en el Tobit, −29.9 en la truncada). La lectura es que el Tobit impone algo que los datos no sostienen: que las mismas variables, con los mismos coeficientes, deciden **si** una mujer trabaja y **cuántas** horas. La educación aumenta mucho la probabilidad de trabajar, pero no las horas de quienes ya trabajan. Es la motivación del **modelo de dos partes** que discuten las notas.